In [ ]:
%pip install numpy
%pip install numpy matplotlib seaborn scikit-learn
%pip install -U replay-trajectory-classification

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../Axona"))
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from utils import prepare_data, maximum_a_posteriori_estimate
from replay_trajectory_classification import SortedSpikesClassifier #from the Frank lab: https://github.com/Eden-Kramer-Lab/replay_trajectory_classification
from replay_trajectory_classification import Environment, RandomWalk, Uniform, estimate_movement_var
import xarray as xr

c:\Users\ajifang\anaconda3\Lib\site-packages\replay_trajectory_classification\likelihoods\multiunit_likelihood.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Online decoding function
## Using decoding method from https://github.com/Eden-Kramer-Lab/replay_trajectory_classification

In [ ]:
def get_online_decoding(filename, Env, session_id, decoding_params):

    #load behavior and spike data
    speed, position, timestamps, SpikeArray,  samplerate, movdir, _ = prepare_data(filename=filename, 
                                                                    session_id=session_id, 
                                                                    Env=Env,
                                                                    upsample_factor = decoding_params['upsampling_scale'],
                                                                    ) #identify cell type here
    
    #get the movement variance
    movement_var = estimate_movement_var(position, samplerate)
    environment = Environment(place_bin_size=decoding_params['place_bin_size'])
    
    is_training = speed > decoding_params['speedThres']
    nfold = decoding_params['nfold']
    #nfold-fold cross validation
    cv = KFold(n_splits=nfold, shuffle=False)

    #init a xarray to store the decoding results
    max_decode_pos = []
    cv_results_all = []
    max_pos_list = []         # store MAP decoded positions for each fold
    test_ind_list = []        # store cv_test_ind for each fold

    for fold_ind, (cv_train_ind, cv_test_ind) in enumerate(cv.split(np.arange(len(position)))):
        
        print(f'Cross-validated the sortedspike decoder: {fold_ind} fold')
        position_train = position[cv_train_ind]
        spikes_train = SpikeArray[:, cv_train_ind]

        spikes_test = SpikeArray[:, cv_test_ind]
        
        #build the decoder using training data
        continuous_transition_types = [[RandomWalk(movement_var=(movement_var[0,0]+movement_var[1,1]/2)*30),  Uniform()],
                                        [Uniform(), Uniform()],
                                    ]

        cv_classifier = SortedSpikesClassifier(
            environments=environment,
            continuous_transition_types=continuous_transition_types,
            sorted_spikes_algorithm='spiking_likelihood_kde',
            sorted_spikes_algorithm_params={'position_std': decoding_params['position_std'],
                                        }
        )


        print("\n===== DEBUG =====")
        print("Animal file:", filename)
        print("Fold:", fold_ind)

        print("position_train shape:", position_train.shape)
        print("spikes_train shape:", spikes_train.shape)

        print(
            "running training bins:",
            np.sum(is_training[cv_train_ind])
        )

        print(
            "total training bins:",
            len(cv_train_ind)
        )

        print(
            "cells with spikes:",
            np.sum(np.nansum(spikes_train, axis=1) > 0)
        )

        print(
            "total spikes:",
            np.nansum(spikes_train)
        )

        print(
            "position finite:",
            np.all(np.isfinite(position_train))
        )

        print(
            "spikes finite:",
            np.all(np.isfinite(spikes_train))
        )

        #encoding using the train data
        cv_classifier.fit(position_train, spikes_train.T, is_training=is_training[cv_train_ind])


        print("fit completed")

        print(
            "place_fields_ type:",
            type(getattr(cv_classifier, "place_fields_", None))
        )

        print(
            "place_fields_:",
            getattr(cv_classifier, "place_fields_", None)
        )
        
        #decoding using the test data
        cv_results = cv_classifier.predict(spikes_test.T, time=timestamps[cv_test_ind], use_gpu=False)
        cv_results_all.append(cv_results.acausal_posterior)

        #get the most likely decoded position at each time bin by marginalizing out states
        summed_posterior = cv_results.acausal_posterior.sum('state')
        max_pos = maximum_a_posteriori_estimate(summed_posterior)
        
        #append the max_pos to max_decode_pos
        max_decode_pos.append(max_pos)
        test_ind_list.append(cv_test_ind)


    # save run_disterror here
    #concatenate the max_decode_pos along the time axis
    max_decode_pos = xr.concat(max_decode_pos, dim='time')
    
    max_decode_pos_np = np.array([list(i) for i in max_decode_pos.values])
    #calculate the distance between max_decode_pos and position at each time point
    disterror = np.linalg.norm(max_decode_pos_np - position, axis=1)

    #calculate online decoding error during running
    run_disterror = disterror[is_training]
    
    return  cv_results_all  #run_disterror   

# Run online decoding

In [ ]:
filenames = [
             r"C:\decode\m20_alldays.mat",
            ] 

Env = 'R'  # VR or R
decoding_params = {
    'speedThres': 4,  # Speed threshold for running vs. immobile
    'upsampling_scale': 4, # Upsampling scale for position and spikes. 1 corresponds to 50 hz, 10 correspond to 500 hz...
    'nfold': 10, # n-fold cross validation
    'position_std': 3, #float or array_like, shape (n_position_dims,) Amount of smoothing for position when building the encoding model. Standard deviation of kernel.
    'place_bin_size': 2, #size of the place bin, bin number = env_size/place_bin_size
}


for filename in filenames:
    print(f'Processing file: {filename}')

    if filename == r"C:\decode\m20_alldays.mat":
        Days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 

    else:
        raise ValueError("Unknown filename")


    for session_id, day_id in enumerate(Days):
        
        print(f'Processing Day {day_id}')
    
        from pathlib import Path

        out_dir = Path(r"C:\decode1")
        out_dir.mkdir(parents=True, exist_ok=True)

        base = Path(filename).stem   #  os.path.basename(...).split(".")[0] 
        save_path = out_dir / f"{base}_{Env}_Day{day_id}_online_decoding_error.npy"
        
        if os.path.exists(save_path):
            print(f"Day {day_id} already processed, skipping...")
            continue    

        online_decoding_cv_results = get_online_decoding(filename = filename,     #online_decoding_error
                                                    Env = Env, 
                                                    session_id = session_id, 
                                                    decoding_params = decoding_params)

        from pathlib import Path
        import pickle

        pkl_path = Path(save_path).with_suffix(".pkl")
        with open(pkl_path, "wb") as f:
            pickle.dump(online_decoding_cv_results, f)
            